# CACEIS Human Capital Value Pipeline

Ce notebook implémente une pipeline complète et défendable pour estimer la **valeur humaine au travail** selon la méthodologie **CACEIS - Alberthon**.

## Objectifs
- Charger et structurer les données RH, performance, formation et absentéisme.
- Traduire la méthodologie théorique en **proxies opérationnels** avec les données disponibles.
- Calculer les composantes `Q`, `B`, `R`, `E`, `Attrition Risk`, `Succession Score` puis le score composite `HCV`.
- Produire des **KPI d'analyse** et des **visualisations Plotly**.
- Entraîner un modèle simple pour **estimer le HCV et le segment HCV** à partir des variables observables.

## Formule cible

$$
HCV = (Q + B + R) \times E \times (1 - Attrition\ Risk) + Succession\ Score
$$

## Important
Cette implémentation reste une **approximation analytique**: certaines briques théoriques (OKR, eNPS, network centrality, benchmark salarial externe, turnover historique multi-annuel) ne sont pas présentes dans les données fournies. Elles sont donc remplacées ici par des **proxies explicitement documentés**.


## 1. Hypothèses de mapping méthodologique

| Composante | Variable recherchée | Présente ? | Remplacement si non | Fichier source | Colonne(s) utilisée(s) |
|---|---|---|---|---|---|
| `Q` | Education & Certifications | Oui | - | `HR Data/Data.xlsx`, `Training/Training_Records_Unnamed.xlsx` | `DEGREE_LEVEL_GROUP_LABEL_EN`, `Certifications` |
| `Q` | OKR Achievement | Non | Note de performance | `HR Data/20240222 - CACEIS Notes evaluation 2023.xlsx`, `HR Data/20250218 - Stats CACEIS EAE EP 18-02-2025 Version Définitive cloture.xlsx`, `HR Data/2025 - Stats CACEIS EAE EP fichier de travail - Vretraitement.xlsx` | `Note`, `Note de performance` |
| `Q` | Time-to-Productivity at hire | Non | Non retenu dans la formule finale pour éviter le double counting avec l'attrition et la succession | - | - |
| `B` | Network centrality / collaboration frequency | Non | Taux de complétion formation | `Training/Training_Records_Unnamed.xlsx`, `Training/Quick_Review_Unnamed.xlsx`, `Training/Cold_Review_Unnamed.xlsx` | `Status`, `Statut` |
| `B` | Project velocity / delivery targets | Non | Présence + mobilité contexte | `HR Data/20260121 - Absentéisme_-_détail_affectation_-_Bilan_social 2025.xlsx`, `HR Data/Data.xlsx` | `Jours Ouvrés Absence`, `Jours Ouvrables Absence`, `taux mob_TO FR` |
| `B` | Presence & mobility | Partiellement | Mobilité contexte pour la France si donnée individuelle absente | `HR Data/20260121 - Absentéisme_-_détail_affectation_-_Bilan_social 2025.xlsx`, `HR Data/Data.xlsx` | `Jours Ouvrés Absence`, `Jours Ouvrables Absence`, `taux mob_TO FR` |
| `R` | Role scarcity | Oui | - | `HR Data/Data.xlsx` | `POSTE_LABEL_LOCAL` |
| `R` | Salary benchmark ratio | Partiellement | Proxy contrat / séniorité si benchmark non disponible | `HR Data/Data.xlsx` | `Compensation Data FR`, `Compensation Data LU`, `CONTRACT_GROUP_LABEL_EN` |
| `R` | Skill inflation factor | Non | Non retenu dans la formule finale pour éviter le double counting avec `Q` | - | - |
| `E` | eNPS | Non | Participation / contexte d'engagement | `Governance/Baromètre Diversité et Inclusion CACEIS - France.pdf`, `Governance/Baromètre Diversité et Inclusion CACEIS - Luxembourg.pdf` | taux de participation, contexte pays |
| `E` | Sentiment trend | Partiellement | Notes et réponses qualitatives formation | `Training/Quick_Review_Unnamed.xlsx`, `Training/Cold_Review_Unnamed.xlsx` | `Note générale`, `Je recommanderais cette formation à un collègue`, réponses qualitatives |
| `E` | Competency coverage / role requirements | Non | Non retenu dans la formule finale pour éviter le double counting avec la composante comportementale / formation | - | - |
| `E` | Engagement / inclusion context | Partiellement | Scores D&I / participation pays + contexte RH | `Governance/Baromètre Diversité et Inclusion CACEIS - France.pdf`, `Governance/Baromètre Diversité et Inclusion CACEIS - Luxembourg.pdf`, `HR Data/Data.xlsx` | score inclusion, taux de participation, variables contexte pays/période |
| `Attrition Risk` | Historical turnover model | Non | Proxy faible ancienneté + absence + turnover contexte | `HR Data/Data.xlsx`, absentéisme, mobilité FR | `DATE_ENTRY_CACEIS`, jours d'absence, `taux_turnover` |
| `Succession Score` | Succession planning / knowledge transfer records | Non | Proxy ancienneté au poste + profondeur d'apprentissage | `HR Data/Data.xlsx`, fichiers training | `DATE_ENTRY_POSTE`, `Total_Training_Hours` |


## Lecture du notebook

Ce notebook est structuré pour la soutenance.

- Les cellules techniques de préparation des données sont conservées pour la traçabilité, mais peuvent être masquées dans l'interface.
- Les sections à commenter en priorité sont les hypothèses, les KPI, les graphiques et l'interprétation du modèle.
- L'objectif n'est pas de détailler chaque ligne de code, mais de montrer la logique méthodologique et les résultats.


In [37]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.io as pio
from IPython.display import display

from human_capital_pipeline import REFERENCE_DATE

pd.set_option('display.max_columns', 120)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')
pio.renderers.default = 'notebook_connected'


In [38]:
from human_capital_pipeline import (
    load_sources, summarize_loaded_sources,
    build_base_snapshot, build_employee_aggregates, build_analysis_df,
    apply_training_missing_strategy, compare_training_strategies,
    compute_hcv_scores, build_summary, build_core_kpis, build_segment_kpis,
    train_hcv_models, build_feature_importance,
    score_new_employee, export_outputs,
)

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / 'caceis-data-provided' / 'Sujet Alberthon'
OUTPUT_DIR = BASE_DIR / 'outputs_hcv_notebook'
OUTPUT_DIR.mkdir(exist_ok=True)

print('Base dir   :', BASE_DIR)
print('Data dir   :', DATA_DIR)
print('Output dir :', OUTPUT_DIR)
print('Reference  :', REFERENCE_DATE.date())


Base dir   : /home/nathan/Téléchargements/caceis
Data dir   : /home/nathan/Téléchargements/caceis/caceis-data-provided/Sujet Alberthon
Output dir : /home/nathan/Téléchargements/caceis/outputs_hcv_notebook
Reference  : 2026-04-28


## 2. Sources mobilisées

In [39]:
sources = load_sources(DATA_DIR)
source_overview = summarize_loaded_sources(sources)
source_overview


,source,rows,columns,unique_ids
0,HR master,275609,15,12711
1,Performance ratings,2544,5,2544
2,EAE definitive,5766,32,3765
3,EAE working,7738,30,3911
4,Absenteeism,127825,46,2264
5,Training records,14943,12,2482
6,Quick review,9706,23,1912
7,Cold review,8647,23,1846
8,Lux absence context,54910,11,2322


## 3. Base RH de référence

In [40]:
base_snapshot = build_base_snapshot(
    sources['master'],
    sources['mobility_fr_raw'],
    sources['absence_fr_context_raw'],
    sources['absence_lu_context'],
    DATA_DIR / 'HR Data' / 'Data.xlsx',
)

print('Employees in latest snapshot:', base_snapshot['employee_id'].nunique())
print('Compensation benchmark match rate:', round(base_snapshot['benchmark_total_comp'].notna().mean(), 3))
base_snapshot[['employee_id', 'period', 'country', 'contract_type', 'degree_level', 'role', 'tenure_caceis_years', 'tenure_position_years']].head()


Employees in latest snapshot: 12711
Compensation benchmark match rate: 0.409


,employee_id,period,country,contract_type,degree_level,role,tenure_caceis_years,tenure_position_years
0,ANON_65X48X48X48X48X50X51X52X50X55X52X,2023-02-01,France,Permanent contract,< Bac,TREASURY CASH MANAGEMENT OFFICER,19.825,7.236
1,ANON_65X48X48X48X48X50X51X52X57X50X56X,2025-05-01,France,Permanent contract,< Bac,WORKS COUNCIL ASSISTANT,19.877,7.236
2,ANON_65X48X48X48X48X50X51X54X48X55X50X,2025-12-01,France,Permanent contract,>=Bac+5,CHIEF COMPLIANCE OFFICER,10.073,0.824
3,ANON_65X48X48X48X48X50X51X54X50X56X56X,2023-08-01,France,Permanent contract,Not specified,CHIEF INFORMATION SECURITY OFFICER,11.992,7.236
4,ANON_65X48X48X48X48X50X51X55X53X55X53X,2025-12-01,France,Permanent contract,>=Bac+5,GLOBAL HEAD,6.574,0.824


## 4. Agrégats analytiques

In [41]:
perf_agg, eae_agg, train_agg, quick_agg, cold_agg, abs_agg = build_employee_aggregates(
    sources['perf'],
    sources['eae'],
    sources['eae_working'],
    sources['train_records'],
    sources['quick'],
    sources['cold'],
    sources['abs_df'],
)

prep_overview = pd.DataFrame([
    {'source': 'Performance agg', 'rows': len(perf_agg), 'unique_ids': perf_agg['employee_id'].nunique()},
    {'source': 'EAE agg', 'rows': len(eae_agg), 'unique_ids': eae_agg['employee_id'].nunique()},
    {'source': 'Training agg', 'rows': len(train_agg), 'unique_ids': train_agg['employee_id'].nunique()},
    {'source': 'Quick agg', 'rows': len(quick_agg), 'unique_ids': quick_agg['employee_id'].nunique()},
    {'source': 'Cold agg', 'rows': len(cold_agg), 'unique_ids': cold_agg['employee_id'].nunique()},
    {'source': 'Absence agg', 'rows': len(abs_agg), 'unique_ids': abs_agg['employee_id'].nunique()},
])
prep_overview


,source,rows,unique_ids
0,Performance agg,2544,2544
1,EAE agg,4260,4260
2,Training agg,2482,2482
3,Quick agg,1912,1912
4,Cold agg,1846,1846
5,Absence agg,2264,2264


## 5. Table analytique finale

In [42]:
df_raw = build_analysis_df(base_snapshot, perf_agg, eae_agg, train_agg, quick_agg, cold_agg, abs_agg)
strategy_comparison = compare_training_strategies(df_raw)

df = compute_hcv_scores(apply_training_missing_strategy(df_raw, 'knn'))

display(strategy_comparison)
df[['employee_id', 'country', 'contract_type', 'degree_level', 'role', 'performance_rating', 'training_hours', 'absence_days', 'HCV']].head()


,strategie,population,hcv_median,hcv_mean,part_high_elite_pct,engagement_mean,training_hours_median
0,Observé uniquement,1050,48.820,48.700,50.000,0.562,69.000
1,Médiane imputée,12711,49.370,48.420,51.000,0.594,26.500
2,KNN imputée,12711,42.110,43.210,51.000,0.552,17.780


,employee_id,country,contract_type,degree_level,role,performance_rating,training_hours,absence_days,HCV
0,ANON_65X48X48X48X48X50X51X52X50X55X52X,France,Permanent contract,< Bac,TREASURY CASH MANAGEMENT OFFICER,3.000,76.110,0.000,46.440
1,ANON_65X48X48X48X48X50X51X52X57X50X56X,France,Permanent contract,< Bac,WORKS COUNCIL ASSISTANT,4.000,76.712,0.000,71.653
2,ANON_65X48X48X48X48X50X51X54X48X55X50X,France,Permanent contract,>=Bac+5,CHIEF COMPLIANCE OFFICER,4.000,63.000,28.000,89.385
3,ANON_65X48X48X48X48X50X51X54X50X56X56X,France,Permanent contract,Not specified,CHIEF INFORMATION SECURITY OFFICER,3.000,43.957,0.000,54.294
4,ANON_65X48X48X48X48X50X51X55X53X55X53X,France,Permanent contract,>=Bac+5,GLOBAL HEAD,3.000,14.000,34.500,45.771


## 6. Construction du score HCV

In [43]:
score_cols = ['Q_qualification', 'B_behavioural', 'R_rarity', 'E_engagement', 'attrition_risk', 'succession_score', 'HCV']
print("Hypothèse retenue pour le scoring principal : imputation KNN des variables de formation manquantes.")
print("Les variantes 'observé uniquement' et 'médiane imputée' sont conservées comme tests de robustesse dans le tableau de comparaison ci-dessus.")
df[score_cols].describe().T


Hypothèse retenue pour le scoring principal : imputation KNN des variables de formation manquantes.
Les variantes 'observé uniquement' et 'médiane imputée' sont conservées comme tests de robustesse dans le tableau de comparaison ci-dessus.


,count,mean,std,min,25%,50%,75%,max
Q_qualification,"12,711.000",0.469,0.064,0.260,0.420,0.465,0.465,0.748
B_behavioural,"12,711.000",0.812,0.068,0.350,0.788,0.812,0.825,1.000
R_rarity,"12,711.000",0.284,0.143,0.090,0.151,0.269,0.402,0.970
E_engagement,"12,711.000",0.552,0.041,0.406,0.529,0.561,0.581,0.606
attrition_risk,"12,711.000",0.452,0.085,0.000,0.417,0.485,0.506,0.716
succession_score,"12,711.000",0.275,0.108,0.000,0.206,0.271,0.340,0.607
HCV,"12,711.000",43.209,19.855,0.000,29.824,42.110,55.749,100.000


## 7. KPI et lecture business

In [44]:
summary = build_summary(df)
core_kpis = build_core_kpis(df)

print('5 KPI business retenus :')
display(core_kpis)

print('Résumé complémentaire :')
summary


5 KPI business retenus :


,kpi,value,why_it_matters
0,HCV median,42.110,Niveau typique de valeur humaine dans l'organi...
1,Part High + Elite,51.000,Poids des profils les plus créateurs de valeur.
2,Talents critiques à risque,9.800,Part des profils à forte valeur déjà exposés a...
3,Engagement moyen (E),0.552,Mesure la qualité du lien entre capital humain...
4,Rareté moyenne (R),0.284,Indique à quel point les profils sont difficil...


Résumé complémentaire :


{'employees': 12711,
 'countries': 18,
 'roles': 514,
 'avg_hcv': 43.21,
 'median_hcv': 42.11,
 'avg_attrition_risk': 0.452,
 'avg_training_hours': 29.81,
 'comp_benchmark_coverage': 0.409,
 'context_inclusion_coverage': 0.553,
 'context_mobility_coverage': 0.29,
 'context_absence_coverage': 0.553,
 'performance_coverage': 1.0,
 'absence_coverage': 0.178,
 'training_coverage': 1.0}

In [45]:
segment_kpis = build_segment_kpis(df)

print('Comparaison des 2 stratégies de traitement des données formation manquantes :')
display(strategy_comparison)

print('Lecture par segment HCV :')
segment_kpis


Comparaison des 2 stratégies de traitement des données formation manquantes :


,strategie,population,hcv_median,hcv_mean,part_high_elite_pct,engagement_mean,training_hours_median
0,Observé uniquement,1050,48.820,48.700,50.000,0.562,69.000
1,Médiane imputée,12711,49.370,48.420,51.000,0.594,26.500
2,KNN imputée,12711,42.110,43.210,51.000,0.552,17.780


Lecture par segment HCV :


,HCV_segment,employees,avg_hcv,avg_q,avg_b,avg_r,avg_e,avg_attrition,avg_tenure,avg_training_hours
1,Elite,3180,69.228,0.500,0.819,0.373,0.572,0.420,9.795,60.468
2,High,3306,48.163,0.478,0.815,0.303,0.551,0.444,7.108,34.249
3,Medium,3047,36.020,0.456,0.818,0.267,0.547,0.464,4.890,16.491
0,Development Potential,3178,18.913,0.441,0.794,0.190,0.539,0.479,4.388,7.300


## 8. Visualisations Plotly

In [46]:
fig = px.histogram(
    df,
    x='HCV',
    nbins=40,
    color='HCV_segment',
    title='Distribution du Human Capital Value (HCV)',
    opacity=0.8,
    marginal='box',
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.update_layout(bargap=0.05, template='plotly_white')
fig.show()
print("Lecture: ce graphe montre la distribution globale du HCV. Une concentration au centre signifie que la majorité des collaborateurs ont un profil relativement proche, tandis que la queue haute correspond aux profils combinant qualification, stabilité et rareté plus fortes.")


Lecture: ce graphe montre la distribution globale du HCV. Une concentration au centre signifie que la majorité des collaborateurs ont un profil relativement proche, tandis que la queue haute correspond aux profils combinant qualification, stabilité et rareté plus fortes.


In [47]:
top_countries = df['country'].value_counts().head(8).index
fig = px.box(
    df[df['country'].isin(top_countries)],
    x='country',
    y='HCV',
    color='country',
    title='HCV par pays',
    points='outliers',
    color_discrete_sequence=px.colors.qualitative.Bold,
)
fig.update_layout(template='plotly_white', showlegend=False)
fig.show()
print("Lecture: ce boxplot compare les niveaux de HCV par pays. On regarde surtout la médiane, l'étendue et les outliers pour voir si certains pays concentrent davantage de profils très valorisés ou, au contraire, plus homogènes.")
print("Attention d'interprétation: la comparaison pays dépend aussi de la couverture de données. La France et le Luxembourg disposent d'un benchmark marché réel, alors que plusieurs autres pays reposent davantage sur des proxies; un écart pays peut donc refléter à la fois une réalité RH et une différence de couverture analytique.")


Lecture: ce boxplot compare les niveaux de HCV par pays. On regarde surtout la médiane, l'étendue et les outliers pour voir si certains pays concentrent davantage de profils très valorisés ou, au contraire, plus homogènes.
Attention d'interprétation: la comparaison pays dépend aussi de la couverture de données. La France et le Luxembourg disposent d'un benchmark marché réel, alors que plusieurs autres pays reposent davantage sur des proxies; un écart pays peut donc refléter à la fois une réalité RH et une différence de couverture analytique.


In [48]:
fig = px.scatter(
    df,
    x='Q_qualification',
    y='E_engagement',
    color='HCV_segment',
    size='training_hours_viz',
    hover_data=['employee_id', 'country', 'contract_type', 'role'],
    title='Lecture croisée: Qualification vs Engagement',
    color_discrete_sequence=px.colors.qualitative.Safe,
)
fig.update_layout(template='plotly_white')
fig.show()
print("Lecture: ce nuage croise qualification et engagement. En haut à droite, on retrouve les profils les plus favorables; en bas à gauche, les profils à développer. La taille des bulles aide à voir si le volume de formation accompagne réellement ces profils.")


Lecture: ce nuage croise qualification et engagement. En haut à droite, on retrouve les profils les plus favorables; en bas à gauche, les profils à développer. La taille des bulles aide à voir si le volume de formation accompagne réellement ces profils.


In [49]:
corr_cols = [
    'HCV', 'Q_qualification', 'B_behavioural', 'R_rarity', 'E_engagement',
    'attrition_risk', 'succession_score', 'tenure_caceis_years',
    'performance_rating', 'training_hours', 'absence_days'
]
corr = df[corr_cols].corr(numeric_only=True)
fig = px.imshow(
    corr,
    text_auto='.2f',
    aspect='auto',
    color_continuous_scale='RdBu_r',
    title='Carte de corrélation des variables clés'
)
fig.update_layout(template='plotly_white')
fig.show()
print("Lecture: cette heatmap mesure les liens statistiques entre variables. Les corrélations positives fortes indiquent des variables qui évoluent ensemble; les corrélations négatives marquées signalent plutôt des mécanismes de compensation ou de risque.")


Lecture: cette heatmap mesure les liens statistiques entre variables. Les corrélations positives fortes indiquent des variables qui évoluent ensemble; les corrélations négatives marquées signalent plutôt des mécanismes de compensation ou de risque.


In [50]:
top_roles = (
    df.groupby('role', as_index=False)
    .agg(avg_hcv=('HCV', 'mean'), employees=('employee_id', 'count'))
    .query('employees >= 20')
    .sort_values('avg_hcv', ascending=False)
    .head(15)
)
fig = px.bar(
    top_roles.sort_values('avg_hcv'),
    x='avg_hcv',
    y='role',
    color='employees',
    orientation='h',
    title='Top 15 rôles par HCV moyen (min 20 employés)',
    color_continuous_scale='Tealgrn'
)
fig.update_layout(template='plotly_white')
fig.show()
print("Lecture: ce classement compare les rôles ayant le HCV moyen le plus élevé parmi les rôles assez représentés. Il permet d'identifier les postes où la combinaison rareté, stabilité et capital humain semble la plus forte.")


Lecture: ce classement compare les rôles ayant le HCV moyen le plus élevé parmi les rôles assez représentés. Il permet d'identifier les postes où la combinaison rareté, stabilité et capital humain semble la plus forte.


## 9. Lecture prédictive du HCV

In [51]:
model_bundle = train_hcv_models(df)
reg_model = model_bundle['reg_model']
clf_model = model_bundle['clf_model']
model_metrics = model_bundle['model_metrics']
model_metrics


,model,r2,rmse,mae,accuracy
0,HCV regressor,0.941,4.703,2.974,NaN
1,HCV segment classifier,NaN,NaN,NaN,0.735


In [52]:
feature_importance, family_importance = build_feature_importance(reg_model)

# Lecture importante: le HCV est un score construit à partir d'un petit nombre de briques.
# Il est donc normal que quelques variables dominent fortement l'importance du modèle,
# tandis que beaucoup de variables one-hot ou secondaires restent proches de 0.

top_features = feature_importance.head(20).copy()
significant_features = feature_importance[feature_importance['importance'] >= 0.005].copy()

fig = px.bar(
    top_features.sort_values('importance'),
    x='importance',
    y='feature',
    orientation='h',
    title='Top 20 drivers du HCV estimé par le modèle',
    color='importance',
    color_continuous_scale='Blues'
)
fig.update_layout(template='plotly_white')
fig.show()
print("Lecture: ce premier graphe détaille les variables individuelles les plus contributives pour reconstituer le HCV. Si quelques barres dominent, cela signifie que certaines dimensions expliquent l'essentiel de la variation du score.")

fig = px.bar(
    significant_features.sort_values('importance'),
    x='importance',
    y='feature',
    orientation='h',
    title='Variables avec importance materielle (>= 0.5%)',
    color='importance',
    color_continuous_scale='Tealgrn'
)
fig.update_layout(template='plotly_white')
fig.show()
print("Lecture: cette vue enlève les micro-effets pour ne garder que les variables réellement matérielles. Elle est plus lisible pour l'oral, car elle montre les facteurs qui pèsent vraiment dans l'estimation.")

fig = px.bar(
    family_importance.sort_values('importance'),
    x='importance',
    y='family',
    orientation='h',
    title='Importance du modele par famille de variables',
    color='importance',
    color_continuous_scale='Oranges'
)
fig.update_layout(template='plotly_white')
fig.show()
print("Lecture: ce dernier graphe regroupe les importances par famille. C'est la meilleure synthèse pour expliquer le modèle: on voit immédiatement si le HCV est surtout piloté par l'ancienneté, la formation, le marché, la qualification ou le risque.")

family_importance


Lecture: ce premier graphe détaille les variables individuelles les plus contributives pour reconstituer le HCV. Si quelques barres dominent, cela signifie que certaines dimensions expliquent l'essentiel de la variation du score.


Lecture: cette vue enlève les micro-effets pour ne garder que les variables réellement matérielles. Elle est plus lisible pour l'oral, car elle montre les facteurs qui pèsent vraiment dans l'estimation.


Lecture: ce dernier graphe regroupe les importances par famille. C'est la meilleure synthèse pour expliquer le modèle: on voit immédiatement si le HCV est surtout piloté par l'ancienneté, la formation, le marché, la qualification ou le risque.


,family,importance
6,Formation,0.468
1,Anciennete,0.348
3,Capital humain,0.086
7,Marche / contrat,0.021
0,Absenteisme,0.021
11,Type de contrat,0.017
8,Pays,0.011
5,Entite,0.008
9,Performance,0.008
10,Role,0.007


## 10. Exemple de scoring d'un nouvel employé

In [53]:
score_new_employee(
    df=df,
    reg_model=reg_model,
    clf_model=clf_model,
    tenure_caceis_years=3,
    tenure_position_years=2,
    performance_rating=3.8,
    training_hours=35,
    training_events=4,
    training_completion_rate=0.9,
    training_sentiment_score=0.85,
    absence_days=4,
    certification_rate=0.5,
    contract_type='CDI',
    degree_level='Master',
    role='FUND ACCOUNTANT',
)


predicted_hcv       65.090
predicted_segment    Elite
dtype: object

## 11. Exports

In [54]:
dataset_path, metadata_path = export_outputs(df, OUTPUT_DIR, summary)

print('Exports written to', OUTPUT_DIR)
print('CSV :', dataset_path)
print('JSON:', metadata_path)


Exports written to /home/nathan/Téléchargements/caceis/outputs_hcv_notebook
CSV : /home/nathan/Téléchargements/caceis/outputs_hcv_notebook/caceis_hcv_dataset.csv
JSON: /home/nathan/Téléchargements/caceis/outputs_hcv_notebook/caceis_hcv_metadata.json


## 12. Points d'attention pour la soutenance

- Le notebook respecte la structure théorique du document `Caceis - Alberthon.pdf`.
- Les composants non observables dans les données fournies sont remplacés par des **proxies explicitement défendus**.
- Le score HCV est donc une **estimation analytique de la valeur humaine au travail**, et non une mesure absolue.
- Les prochaines améliorations naturelles seraient: benchmark salarial externe, historique de turnover multi-annuel, enquêtes pulse RH/eNPS, données de mobilité et de collaboration interne.
